# 📄 PRUEBA MÓDULO 3 — UrbanRent Analytics

**Caso de estudio:** Pipeline de análisis sobre datos de reservas, apartamentos y propietarios.

---

## ⚠️ Compatibilidad Almond + Spark — Lee esto primero

La celda de diagnóstico confirma que tu kernel tiene **`scala-library 2.13.14`**.
Spark 4.1.1 fue compilado contra **2.13.15** → incompatibilidad binaria → `NoSuchMethodError: MurmurHash3`.

### Dos caminos:

| | Opción A — Sin reinstalar *(este notebook)* | Opción B — Upgrade permanente |
|---|---|---|
| **Qué hace** | Usa Spark **4.0.0** (compilado contra 2.13.14) | Reinstala Almond con Scala 2.13.15 y usa Spark 4.1.1 |
| **Comando** | — *(ya aplicado en Celda 0)* | Ver sección abajo |
| **Reiniciar kernel** | No | Sí |

### Opción B — Reinstalar Almond (Windows, PowerShell)
```powershell
# 1. Descarga Coursier si no lo tienes
Invoke-WebRequest -Uri https://github.com/coursier/launchers/raw/master/cs-x86_64-pc-win32.zip -OutFile cs.zip
Expand-Archive cs.zip .; .\cs setup

# 2. Instala Almond con Scala 2.13.15
cs launch almond:0.14.0-RC14 --scala 2.13.15 -- --install --force

# 3. Reinicia Jupyter y selecciona el kernel 'Scala 2.13'
```
Tras esto podrás usar `import $ivy.\`org.apache.spark::spark-sql:4.1.1\`` sin errores.

> **Este notebook usa la Opción A** (Spark 4.0.0 + tu Almond actual). Ejecuta las celdas en orden.

---

## 📦 Paso previo — Creación de los ficheros de datos


In [ ]:
// ── Generación de ficheros de datos ─────────────────────────────────────────
import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets

val reservasCSV =
  """id_reserva,id_apartamento,fecha_entrada,fecha_salida,num_huespedes,precio_noche,canal,valoracion,ciudad
R001,APT001,2025-01-10,2025-01-13,2,95.0,Airbnb,4.8,Madrid
R002,APT002,2025-01-12,2025-01-15,4,120.0,Booking,4.5,Barcelona
R003,APT003,2025-01-20,2025-01-22,1,75.0,Directo,5.0,Valencia
R004,APT001,2025-02-01,2025-02-05,3,95.0,Airbnb,4.7,Madrid
R005,APT004,2025-02-10,2025-02-12,2,110.0,Booking,4.2,Sevilla
R006,APT002,2025-02-14,2025-02-17,5,120.0,Airbnb,4.9,Barcelona
R007,APT005,2025-02-20,2025-02-23,2,85.0,Directo,4.6,Bilbao
R008,APT003,2025-03-01,2025-03-04,1,75.0,Booking,4.4,Valencia
R009,APT001,2025-03-10,2025-03-14,4,95.0,Airbnb,4.8,Madrid
R010,APT006,2025-03-15,2025-03-17,2,130.0,Directo,5.0,Madrid
R011,APT004,2025-03-20,2025-03-22,3,110.0,Airbnb,4.3,Sevilla
R012,APT007,2025-04-01,2025-04-04,2,90.0,Booking,4.5,Barcelona
R013,APT005,2025-04-08,2025-04-10,1,85.0,Directo,4.7,Bilbao
R014,APT006,2025-04-12,2025-04-16,3,130.0,Airbnb,4.9,Madrid
R015,APT002,2025-04-20,2025-04-23,4,120.0,Booking,4.6,Barcelona
R016,APT008,2025-05-01,2025-05-04,2,70.0,Directo,4.1,Valencia
R017,APT003,2025-05-10,2025-05-13,2,75.0,Airbnb,4.5,Valencia
R018,APT007,2025-05-15,2025-05-18,3,90.0,Booking,4.8,Barcelona
R019,APT001,2025-05-20,2025-05-24,2,95.0,Directo,5.0,Madrid
R020,APT009,2025-05-25,2025-05-28,1,65.0,Airbnb,3.9,Bilbao
R021,APT006,2025-06-01,2025-06-05,4,130.0,Booking,4.7,Madrid
R022,APT010,2025-06-08,2025-06-10,2,80.0,Directo,4.3,Sevilla
R023,APT004,2025-06-15,2025-06-18,3,110.0,Airbnb,4.6,Sevilla
R024,APT008,2025-06-20,2025-06-22,1,70.0,Booking,4.2,Valencia
R025,APT005,2025-06-25,2025-06-28,2,85.0,Directo,4.8,Bilbao
"""

val apartamentosCSV =
  """id_apartamento,tipo,habitaciones,metros_cuadrados,tiene_parking,propietario_id,activo
APT001,Estudio,1,35,false,P01,true
APT002,Piso,3,85,true,P02,true
APT003,Estudio,1,30,false,P01,true
APT004,Piso,2,65,true,P03,true
APT005,Ático,2,70,true,P04,true
APT006,Piso,3,90,true,P02,true
APT007,Estudio,1,28,false,P05,true
APT008,Estudio,1,32,false,P03,false
APT009,Piso,2,55,false,P04,true
APT010,Ático,3,100,true,P05,true
"""

val propietariosJSON =
  """[
  {"propietario_id": "P01", "nombre": "Laura Sánchez", "ciudad": "Madrid", "antiguedad_anios": 5, "tipo_contrato": "Premium", "especialidades": ["Estudio","Piso"]},
  {"propietario_id": "P02", "nombre": "Carlos Méndez", "ciudad": "Barcelona", "antiguedad_anios": 3, "tipo_contrato": "Estándar", "especialidades": ["Piso"]},
  {"propietario_id": "P03", "nombre": "Ana Ferrero", "ciudad": "Sevilla", "antiguedad_anios": 7, "tipo_contrato": "Premium", "especialidades": ["Piso","Ático"]},
  {"propietario_id": "P04", "nombre": "Javier Ruiz", "ciudad": "Bilbao", "antiguedad_anios": 2, "tipo_contrato": "Estándar", "especialidades": ["Ático","Piso"]},
  {"propietario_id": "P05", "nombre": "Marta Gil", "ciudad": "Valencia", "antiguedad_anios": 6, "tipo_contrato": "Premium", "especialidades": ["Estudio","Ático"]}
]
"""

Files.write(Paths.get("reservas.csv"),       reservasCSV.getBytes(StandardCharsets.UTF_8))
Files.write(Paths.get("apartamentos.csv"),   apartamentosCSV.getBytes(StandardCharsets.UTF_8))
Files.write(Paths.get("propietarios.json"),  propietariosJSON.getBytes(StandardCharsets.UTF_8))

println("✅ Ficheros creados: reservas.csv | apartamentos.csv | propietarios.json")

---
### 🔍 Celda de diagnóstico — Versión de Scala en el kernel
Ejecuta esta celda para conocer la versión exacta de `scala-library` cargada por Almond.
La necesitarás para la celda de pin de versión en la Celda 0.

In [ ]:
// ── Diagnóstico: versión de scala-library del kernel ─────────────────────────
// Ejecuta esto PRIMERO para saber qué versión exacta usar en el pin de la Celda 0
val scalaVersion = scala.util.Properties.versionString
val scalaLibVersion = classOf[scala.Option[_]].getProtectionDomain
  .getCodeSource.getLocation.toString

println(s"Scala runtime: $scalaVersion")
println(s"scala-library JAR: $scalaLibVersion")

// También muestra el classpath completo (útil para debug avanzado)
// Descomenta si necesitas verlo:
// System.getProperty("java.class.path").split(':').foreach(println)


---
## ⚙️ Celda 0 — Inicialización del entorno (obligatoria)

In [ ]:
// ── Celda 0 — Inicialización ─────────────────────────────────────────────────
// Spark 4.0.0 — compilado contra scala-library 2.13.14, compatible con este kernel
// (Spark 4.1.x requiere 2.13.15 → NoSuchMethodError con el kernel actual)
import $ivy.`org.apache.spark::spark-core:4.0.0`
import $ivy.`org.apache.spark::spark-sql:4.0.0`

import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
import org.apache.spark.sql.expressions.Window
import org.apache.spark.sql.types._

val spark = SparkSession.builder()
  .appName("UrbanRent_Analytics")
  .master("local[*]")
  .config("spark.sql.shuffle.partitions", "4")
  .config("spark.sql.crossJoin.enabled", "true")
  .getOrCreate()

import spark.implicits._
spark.sparkContext.setLogLevel("ERROR")

println(s"✅ UrbanRent Analytics iniciado — Spark ${spark.version} · Scala ${scala.util.Properties.versionString}")
// Salida esperada: Spark 4.0.0 · Scala version 2.13.14


---
## 🗂️ PARTE 1 — Carga y exploración inicial

### Pregunta 1 — ¿Están bien formateados nuestros datos de reservas?

In [ ]:
// ── Pregunta 1 — Carga y exploración de reservas.csv ────────────────────────
val reservas = spark.read
  .option("header", "true")
  .option("inferSchema", "true")
  .csv("reservas.csv")

println("=== Primeras 8 reservas ===")
reservas.show(8, truncate = false)

println("\n=== Schema inferido ===")
reservas.printSchema()

println("\n=== Estadísticas descriptivas ===")
reservas.describe("precio_noche", "num_huespedes", "valoracion").show(truncate = false)

val totalReservas = reservas.count()
val columnas = reservas.columns.mkString(" | ")
println(s"\nTotal de reservas: $totalReservas")
println(s"Columnas: $columnas")

---
### Pregunta 2 — ¿Es correcto el schema inferido o necesitamos ajustarlo?

In [ ]:
// ── Pregunta 2 — Schema manual con tipos correctos ───────────────────────────
val schemaReservas = StructType(Seq(
  StructField("id_reserva",      StringType,  nullable = true),
  StructField("id_apartamento",  StringType,  nullable = true),
  StructField("fecha_entrada",   DateType,    nullable = true),
  StructField("fecha_salida",    DateType,    nullable = true),
  StructField("num_huespedes",   IntegerType, nullable = true),
  StructField("precio_noche",    DoubleType,  nullable = true),
  StructField("canal",           StringType,  nullable = true),
  StructField("valoracion",      DoubleType,  nullable = true),
  StructField("ciudad",          StringType,  nullable = true)
))

// Recarga con schema explícito — fecha_entrada/salida quedan como DateType
val reservas = spark.read
  .option("header", "true")
  .option("dateFormat", "yyyy-MM-dd")
  .schema(schemaReservas)
  .csv("reservas.csv")

println("=== Schema ajustado manualmente ===")
reservas.printSchema()

println("\n=== Primeras 5 filas con tipos corregidos ===")
reservas.show(5, truncate = false)

---
### Pregunta 3 — ¿Cuántos apartamentos tenemos y cuáles son sus características?

In [ ]:
// ── Pregunta 3 — Carga de apartamentos y propietarios ───────────────────────
val apartamentos = spark.read
  .option("header", "true")
  .option("inferSchema", "true")
  .csv("apartamentos.csv")

val propietarios = spark.read
  .option("multiline", "true")
  .json("propietarios.json")

println("=== Schema apartamentos ===")
apartamentos.printSchema()
println("=== Primeras filas apartamentos ===")
apartamentos.show(truncate = false)
println(s"Total apartamentos: ${apartamentos.count()}")

println("\n=== Schema propietarios ===")
propietarios.printSchema()
println("=== Primeras filas propietarios ===")
propietarios.show(truncate = false)
println(s"Total propietarios: ${propietarios.count()}")

val aptActivos = apartamentos.filter($"activo" === true).count()
println(s"\n✅ Apartamentos activos (activo = true): $aptActivos")

---
## 🔧 PARTE 2 — Operaciones básicas sobre DataFrames

### Pregunta 4 — ¿Qué reservas corresponden a estancias de más de 3 noches?

In [ ]:
// ── Pregunta 4 — Reservas largas (> 3 noches) ────────────────────────────────
val reservasConNoches = reservas
  .withColumn("num_noches",      datediff($"fecha_salida", $"fecha_entrada"))
  .withColumn("ingreso_reserva", $"num_noches" * $"precio_noche")

val reservasLargas = reservasConNoches
  .filter($"num_noches" > 3)
  .select("id_reserva", "ciudad", "canal", "num_noches", "ingreso_reserva")
  .orderBy($"ingreso_reserva".desc)

println("=== Reservas largas (más de 3 noches) ===")
reservasLargas.show(truncate = false)
println(s"Total reservas largas: ${reservasLargas.count()}")

---
### Pregunta 5 — ¿Qué canales de venta tenemos activos y cuál es el precio medio por canal?

In [ ]:
// ── Pregunta 5 — Canales únicos con precio representativo ───────────────────
// dropDuplicates mantiene la primera aparición de cada canal
val canalesUnicos = reservas
  .dropDuplicates("canal")
  .select("canal", "precio_noche")
  .orderBy($"precio_noche".desc)

println("=== Canales únicos (precio_noche de la primera reserva del canal) ===")
canalesUnicos.show(truncate = false)

// Precio medio real por canal (análisis complementario)
println("=== Precio medio real por canal ===")
reservas
  .groupBy("canal")
  .agg(round(avg("precio_noche"), 2).as("precio_medio"))
  .orderBy($"precio_medio".desc)
  .show(truncate = false)

---
### Pregunta 6 — ¿Cuánto ingresaría cada reserva con suplemento del 10% para más de 2 huéspedes?

In [ ]:
// ── Pregunta 6 — Suplemento 10% para reservas con > 2 huéspedes ─────────────
val reservasSuplemento = reservasConNoches
  .withColumn(
    "precio_con_suplemento",
    when($"num_huespedes" > 2, $"precio_noche" * 1.10)
      .otherwise($"precio_noche")
  )
  .withColumn("ingreso_estimado", $"num_noches" * $"precio_con_suplemento")
  .select(
    "id_reserva", "ciudad", "num_huespedes",
    "precio_noche", "precio_con_suplemento", "ingreso_estimado"
  )
  .orderBy($"ingreso_estimado".desc)

println("=== Reservas con suplemento por huéspedes (ordenadas por ingreso estimado) ===")
reservasSuplemento.show(truncate = false)

---
### Pregunta 7 — ¿Existen ciudades con reservas que no aparecen en propietarios?

In [ ]:
// ── Pregunta 7 — Comparativa de ciudades entre fuentes ──────────────────────
// Renombramos ciudad en propietarios para evitar ambigüedades en joins futuros
val propietarios = spark.read
  .option("multiline", "true")
  .json("propietarios.json")
  .withColumnRenamed("ciudad", "ciudad_propietario")

println("=== Ciudades únicas en reservas ===")
reservas.select("ciudad").distinct().orderBy("ciudad").show(truncate = false)

println("=== Ciudades únicas en propietarios ===")
propietarios.select("ciudad_propietario").distinct().orderBy("ciudad_propietario").show(truncate = false)

println("""
📝 Análisis:
  - Las reservas cubren: Madrid, Barcelona, Valencia, Sevilla, Bilbao.
  - Los propietarios residen en las mismas 5 ciudades.
  - No hay ciudades presentes en apartamentos/propietarios que falten en reservas.
""")

---
## 📊 PARTE 3 — Agregaciones y métricas de negocio

### Pregunta 8 — ¿Qué ciudad genera más ingresos para la empresa?

In [ ]:
// ── Pregunta 8 — Ranking de ingresos por ciudad ──────────────────────────────
// Aseguramos que ingreso_reserva existe en el DataFrame base
val reservasBase = reservasConNoches  // ya tiene num_noches e ingreso_reserva

val rankingCiudades = reservasBase
  .groupBy("ciudad")
  .agg(
    count("*").as("num_reservas"),
    round(sum("ingreso_reserva"),   2).as("ingreso_total"),
    round(avg("ingreso_reserva"),   2).as("ticket_medio"),
    round(avg("valoracion"),        2).as("valoracion_media")
  )
  .orderBy($"ingreso_total".desc)

println("=== Ranking de ciudades por ingresos totales ===")
rankingCiudades.show(truncate = false)

---
### Pregunta 9 — ¿Cuál es el canal de venta más rentable por ciudad?

In [ ]:
// ── Pregunta 9 — Canal más rentable por ciudad ───────────────────────────────
val canalPorCiudad = reservasBase
  .groupBy("ciudad", "canal")
  .agg(
    round(sum("ingreso_reserva"), 2).as("ingreso_total"),
    count("*").as("num_reservas")
  )
  .orderBy($"ciudad".asc, $"ingreso_total".desc)

println("=== Canal más rentable por ciudad ===")
canalPorCiudad.show(truncate = false)

---
### Pregunta 10 — ¿Qué apartamento tiene la mejor posición de ingreso dentro de su ciudad?

In [ ]:
// ── Pregunta 10 — Ranking de apartamentos por ciudad (Window) ────────────────
val ingresosPorApt = reservasBase
  .groupBy("id_apartamento", "ciudad")
  .agg(round(sum("ingreso_reserva"), 2).as("ingreso_total"))

val ventanaCiudad = Window
  .partitionBy("ciudad")
  .orderBy($"ingreso_total".desc)

val rankingApt = ingresosPorApt
  .withColumn("ranking_ciudad", rank().over(ventanaCiudad))

println("=== Apartamento número 1 por ingresos en cada ciudad ===")
rankingApt
  .filter($"ranking_ciudad" === 1)
  .orderBy($"ingreso_total".desc)
  .show(truncate = false)

println("=== Ranking completo de apartamentos por ciudad ===")
rankingApt.orderBy("ciudad", "ranking_ciudad").show(truncate = false)

---
### Pregunta 11 — ¿Cuánto se ha facturado por ciudad y canal, con subtotales? (rollup)

In [ ]:
// ── Pregunta 11 — Facturación con subtotales mediante rollup ─────────────────
val factRollup = reservasBase
  .rollup("ciudad", "canal")
  .agg(round(sum("ingreso_reserva"), 2).as("total_facturado"))
  .orderBy($"ciudad".asc_nulls_last, $"canal".asc_nulls_last)

println("=== Facturación con rollup (subtotales por ciudad y gran total) ===")
factRollup.show(50, truncate = false)

println("""
📝 Reflexión sobre las filas nulas en rollup:
  · Fila con canal = null y ciudad presente  → subtotal de ESA CIUDAD sumando todos sus canales.
  · Fila con ciudad = null Y canal = null     → GRAN TOTAL global de toda la facturación.
""")

---
### Pregunta 12 — ¿Podemos ver la facturación como tabla cruzada? (pivot)

In [ ]:
// ── Pregunta 12 — Tabla pivot ciudad × canal ─────────────────────────────────
val tablaPivot = reservasBase
  .groupBy("ciudad")
  .pivot("canal")
  .agg(round(sum("ingreso_reserva"), 2))
  .orderBy("ciudad")

println("=== Tabla pivot: ingresos por ciudad y canal ===")
tablaPivot.show(truncate = false)

println("""
📝 Reflexión sobre los null en la tabla pivot:
  · Un valor null en una celda significa que esa ciudad NO tiene ninguna
    reserva registrada para ese canal concreto en el período analizado.
""")

---
## 🔗 PARTE 4 — Cruce de fuentes de datos (Joins)

> **Preparación:** Se recargan los DataFrames base y se renombran columnas ambiguas.

In [ ]:
// ── Preparación para la sección de Joins ────────────────────────────────────
// Volvemos a cargar apartamentos renombrando 'activo' para evitar conflictos
val apartamentos = spark.read
  .option("header", "true")
  .option("inferSchema", "true")
  .csv("apartamentos.csv")
  .withColumnRenamed("activo", "apt_activo")

// propietarios ya tiene ciudad renombrada a ciudad_propietario (ver P7)
// Nos aseguramos de tenerlo disponible en este scope
val propietarios = spark.read
  .option("multiline", "true")
  .json("propietarios.json")
  .withColumnRenamed("ciudad", "ciudad_propietario")

// reservasBase tiene num_noches e ingreso_reserva
println("✅ DataFrames preparados para la sección de Joins")
println(s"  · reservasBase: ${reservasBase.columns.mkString(", ")}")
println(s"  · apartamentos: ${apartamentos.columns.mkString(", ")}")
println(s"  · propietarios: ${propietarios.columns.mkString(", ")}")

---
### Pregunta 13 — ¿Cuántas noches y a qué precio se ha alquilado cada apartamento? (INNER JOIN)

In [ ]:
// ── Pregunta 13 — INNER JOIN reservas × apartamentos ────────────────────────
val reservasApt = reservasBase
  .join(apartamentos, Seq("id_apartamento"), "inner")
  .select(
    $"id_reserva",
    $"ciudad",          // columna de reservas (no ambigua tras el join por key)
    $"tipo",
    $"habitaciones",
    $"metros_cuadrados",
    $"precio_noche",
    $"ingreso_reserva"
  )
  .orderBy($"ingreso_reserva".desc)

println("=== INNER JOIN: reservas enriquecidas con datos de apartamento ===")
reservasApt.show(truncate = false)

val filasJoin = reservasApt.count()
println(s"Total filas tras INNER JOIN: $filasJoin")
println("✅ 25 filas → no se pierde ninguna reserva (todos los APT existen en el catálogo).")

---
### Pregunta 14 — ¿Hay apartamentos que nunca han tenido ninguna reserva? (LEFT JOIN)

In [ ]:
// ── Pregunta 14 — Apartamentos sin reservas (LEFT JOIN + filtro null) ─────────
val aptSinReservas = apartamentos
  .join(reservasBase, Seq("id_apartamento"), "left")
  .filter($"id_reserva".isNull)
  .select("id_apartamento", "tipo", "habitaciones", "propietario_id")

println("=== Apartamentos sin ninguna reserva ===")
aptSinReservas.show(truncate = false)

println("""
📝 Reflexión:
  · Para ver TODAS las reservas aunque el apartamento ya no exista en el catálogo
    usaríamos un RIGHT JOIN (reservas como tabla izquierda) o un FULL OUTER JOIN.
""")

---
### Pregunta 15 — ¿Qué propietarios tienen al menos un apartamento? (LEFT SEMI JOIN)

In [ ]:
// ── Pregunta 15 — Propietarios con apartamentos en catálogo (SEMI JOIN) ───────
val propietariosActivos = propietarios
  .join(apartamentos, Seq("propietario_id"), "leftsemi")
  .select("propietario_id", "nombre", "tipo_contrato")

println("=== Propietarios con al menos un apartamento en catálogo ===")
propietariosActivos.show(truncate = false)

---
### Pregunta 16 — ¿Qué propietarios no tienen ningún apartamento? (LEFT ANTI JOIN)

In [ ]:
// ── Pregunta 16 — Propietarios sin apartamentos (ANTI JOIN) ──────────────────
val propietariosSinApt = propietarios
  .join(apartamentos, Seq("propietario_id"), "leftanti")
  .select("propietario_id", "nombre", "ciudad_propietario")

println("=== Propietarios SIN ningún apartamento en el catálogo ===")
propietariosSinApt.show(truncate = false)

if (propietariosSinApt.count() == 0)
  println("ℹ️  Todos los propietarios tienen al menos un apartamento registrado.")

---
### Pregunta 17 — Informe consolidado: reservas + apartamento + propietario (JOIN triple)

In [ ]:
// ── Pregunta 17 — JOIN triple: reservas × apartamentos × propietarios ─────────
val informeCompleto = reservasBase
  .join(apartamentos, Seq("id_apartamento"), "inner")
  .join(propietarios, Seq("propietario_id"),  "inner")
  .select(
    $"id_reserva",
    $"ciudad",           // ciudad de la reserva
    $"tipo",
    $"habitaciones",
    $"nombre",           // nombre del propietario
    $"tipo_contrato",
    $"ingreso_reserva"
  )
  .orderBy($"ingreso_reserva".desc)

println("=== Informe consolidado: reservas + apartamento + propietario ===")
informeCompleto.show(truncate = false)
println(s"Total filas: ${informeCompleto.count()}")

---
### Pregunta 18 — ¿Tiene sentido aplicar broadcast en alguno de estos joins?

In [ ]:
// ── Pregunta 18 — Broadcast join: comparativa de planes ──────────────────────

// JOIN sin broadcast
val joinSinBroadcast = reservasBase
  .join(apartamentos, Seq("id_apartamento"), "inner")

println("=" * 60)
println("PLAN FÍSICO — Sin broadcast")
println("=" * 60)
joinSinBroadcast.explain("simple")

// JOIN con broadcast sobre la tabla pequeña (apartamentos)
val joinConBroadcast = reservasBase
  .join(broadcast(apartamentos), Seq("id_apartamento"), "inner")

println("=" * 60)
println("PLAN FÍSICO — Con broadcast(apartamentos)")
println("=" * 60)
joinConBroadcast.explain("simple")

#### 📝 Reflexión — Broadcast join

| Aspecto | Sin broadcast | Con broadcast |
|---|---|---|
| **Tipo de join en el plan** | `SortMergeJoin` (ambas tablas se ordenan y mezclan) | `BroadcastHashJoin` (la tabla pequeña se replica en todos los ejecutores) |
| **Shuffle de red** | Sí — ambas particiones se redistribuyen por clave | No — solo se envía la tabla pequeña una vez a cada nodo |
| **¿Por qué broadcast sobre `apartamentos`?** | `apartamentos` tiene 10 filas (~KB). Cabe fácilmente en memoria de cada executor. `reservas` puede crecer a millones de filas y NO debe replicarse. |

---
## 🔠 PARTE 5 — Funciones integradas y UDFs

### Pregunta 19 — ¿Cómo presentar las fechas de entrada en formato europeo?

In [ ]:
// ── Pregunta 19 — Formateo de fechas (date_format, month, year) ───────────────
val reservasFechasES = reservas
  .withColumn("fecha_entrada_es", date_format($"fecha_entrada", "dd/MM/yyyy"))
  .withColumn("mes_entrada",      month($"fecha_entrada"))
  .withColumn("anio_entrada",     year($"fecha_entrada"))
  .select("id_reserva", "ciudad", "fecha_entrada", "fecha_entrada_es", "mes_entrada", "anio_entrada")

println("=== Fechas de entrada en formato europeo (primeras 6 filas) ===")
reservasFechasES.show(6, truncate = false)

---
### Pregunta 20 — ¿Cuántos días de antelación se hacen las reservas?

In [ ]:
// ── Pregunta 20 — Días de antelación respecto a fecha de referencia ───────────
val fechaRef = to_date(lit("2025-01-01"), "yyyy-MM-dd")

val reservasAntelacion = reservas
  .withColumn("fecha_ref",      fechaRef)
  .withColumn("dias_antelacion", datediff($"fecha_entrada", $"fecha_ref"))
  .select("id_reserva", "ciudad", "fecha_entrada", "dias_antelacion")
  .orderBy($"dias_antelacion".asc)

println("=== Días de antelación respecto al 2025-01-01 (más próximas primero) ===")
reservasAntelacion.show(truncate = false)

---
### Pregunta 21 — ¿Están bien formateados los IDs de reserva? Estandarización

In [ ]:
// ── Pregunta 21 — Limpieza de texto: trim, upper, initcap ────────────────────
val reservasLimpias = reservas
  .withColumn("id_reserva_sucio",   concat($"id_reserva", lit(" ")))  // espacio al final
  .withColumn("id_reserva_limpio",  upper(trim($"id_reserva_sucio")))
  .withColumn("ciudad_normalizada", initcap($"ciudad"))
  .select("id_reserva_sucio", "id_reserva_limpio", "ciudad", "ciudad_normalizada")

println("=== Comparativa ID sucio vs limpio y ciudad normalizada (primeras 5 filas) ===")
reservasLimpias.show(5, truncate = false)

---
### Pregunta 22 — ¿Qué especialidades tienen registradas los propietarios? (explode)

In [ ]:
// ── Pregunta 22 — Explode del array de especialidades ────────────────────────
// Recargamos propietarios con ciudad renombrada (scope local)
val propietariosRaw = spark.read
  .option("multiline", "true")
  .json("propietarios.json")

val especialidadesExploded = propietariosRaw
  .withColumn("especialidad", explode($"especialidades"))
  .select("propietario_id", "nombre", "especialidad")

println("=== Especialidades por propietario (una fila por especialidad) ===")
especialidadesExploded.show(truncate = false)
println(s"Total filas tras explode: ${especialidadesExploded.count()}")

println("\n=== Propietarios por especialidad ===")
especialidadesExploded
  .groupBy("especialidad")
  .count()
  .orderBy($"count".desc)
  .show(truncate = false)

---
### Pregunta 23 — Clasificación de calidad de reservas mediante UDF

In [ ]:
// ── Pregunta 23 — UDF de clasificación de calidad ────────────────────────────
val clasificarCalidadFn: java.lang.Double => String = valoracion =>
  if (valoracion == null)    "Sin valoración"
  else if (valoracion >= 4.8) "Excelente"
  else if (valoracion >= 4.5) "Muy buena"
  else if (valoracion >= 4.0) "Buena"
  else                        "Mejorable"

// Registrar como UDF de Spark Dataset API
val clasificarCalidadUDF = udf(clasificarCalidadFn)

// Registrar también en Spark SQL (para P24)
spark.udf.register("clasificar_calidad", clasificarCalidadFn)

// Aplicar la UDF sobre reservas
val reservasConCalidad = reservas
  .withColumn("calidad_reserva", clasificarCalidadUDF($"valoracion"))

println("=== Reservas con clasificación de calidad ===")
reservasConCalidad
  .select("id_reserva", "ciudad", "valoracion", "calidad_reserva")
  .show(truncate = false)

println("=== Reservas por categoría de calidad ===")
reservasConCalidad
  .groupBy("calidad_reserva")
  .count()
  .orderBy($"count".desc)
  .show(truncate = false)

---
### Pregunta 24 — Informe final consolidado con Spark SQL

In [ ]:
// ── Pregunta 24 — Vista temporal + consulta Spark SQL ────────────────────────
// Construimos el DataFrame enriquecido con todas las columnas necesarias
val reservasEnriquecidas = reservas
  .withColumn("num_noches",      datediff($"fecha_salida", $"fecha_entrada"))
  .withColumn("ingreso_reserva", $"num_noches" * $"precio_noche")
  .withColumn("calidad_reserva", clasificarCalidadUDF($"valoracion"))
  .withColumn("mes_entrada",     month($"fecha_entrada"))

// Registro como vista temporal
reservasEnriquecidas.createOrReplaceTempView("reservas_enriquecidas")
println("✅ Vista temporal 'reservas_enriquecidas' registrada.")

// Consulta SQL del enunciado
val informeSQL = spark.sql("""
  SELECT
    ciudad,
    canal,
    COUNT(*)                              AS num_reservas,
    ROUND(SUM(ingreso_reserva), 2)        AS ingreso_total,
    ROUND(AVG(valoracion), 2)             AS valoracion_media,
    clasificar_calidad(AVG(valoracion))   AS calidad_media
  FROM reservas_enriquecidas
  GROUP BY ciudad, canal
  ORDER BY ingreso_total DESC
""")

println("\n=== Informe final consolidado (Spark SQL) ===")
informeSQL.show(50, truncate = false)

---
## ✅ Verificación final

In [ ]:
// ── Verificación final ───────────────────────────────────────────────────────
println("=" * 60)
println("VERIFICACIÓN FINAL — Caso de Estudio UrbanRent Analytics")
println("=" * 60)

// Recargamos propietarios sin renombrar para el explode de especialidades
val propietariosRawFinal = spark.read
  .option("multiline", "true")
  .json("propietarios.json")

val comprobaciones = Seq(
  ("P1  — Total reservas cargadas (25)",
    reservas.count() == 25L),

  ("P3  — Apartamentos activos (9)",
    spark.read
      .option("header", "true")
      .option("inferSchema", "true")
      .csv("apartamentos.csv")
      .filter($"activo" === true)
      .count() == 9L),

  ("P4  — Reservas largas > 3 noches",
    reservasLargas.count() > 0L),

  ("P8  — Ciudades en el ranking de ingresos (5)",
    reservas.select("ciudad").distinct().count() == 5L),

  ("P13 — INNER JOIN reservas × apartamentos (25 filas)",
    reservasBase.join(
      spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv("apartamentos.csv"),
      Seq("id_apartamento"), "inner"
    ).count() == 25L),

  ("P22 — Explode especialidades propietarios (>= 5 filas)",
    propietariosRawFinal
      .select(explode($"especialidades"))
      .count() >= 5L)
)

comprobaciones.foreach { case (desc, ok) =>
  println(s"${if (ok) "✅ CORRECTO" else "❌ REVISAR"} — $desc")
}

println("=" * 60)